# Exploration OpenAgenda

---

- **Projet 9 :** Concevez et déployez un système RAG
- **Auteur :** Justine Tranchant
- **Date :** mai 2026

---

Objectif : comprendre comment récupérer les événements publics OpenAgenda, tester les filtres utiles et identifier les champs à conserver pour la suite du projet RAG.

Ce notebook correspond à l'étape d'exploration. Il ne construit pas encore le client final ni l'index FAISS.

## Imports et configuration

On utilise `requests` pour appeler l'API, `pandas` pour inspecter les résultats, et `json` pour sauvegarder un échantillon brut.

In [1]:
import json
import sys

import pandas as pd
import requests

# Permet d'importer src.config quand le notebook est exécuté depuis le dossier notebooks/.
sys.path.append("..")

from src.config import (
    OPENAGENDA_BASE_URL,
    OPENAGENDA_CITIES,
    OPENAGENDA_ORDER_BY,
    OPENAGENDA_PAGE_SIZE,
    OPENAGENDA_USEFUL_FIELDS,
    PATHS,
    RAW_OPENAGENDA_SAMPLE_FILENAME,
)

from src.openagenda import build_openagenda_where_clause, save_openagenda_sample

In [2]:
PATHS.data_raw.mkdir(parents=True, exist_ok=True)

OPENAGENDA_BASE_URL

'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records'

## Villes ciblées

Pour le POC, on limite volontairement la zone géographique au Bassin d'Arcachon et à quelques villes proches.

Ce choix permet d'obtenir un jeu de données local, cohérent avec un assistant de recommandation d'événements.

In [3]:
OPENAGENDA_CITIES

['Arcachon',
 'La Teste-de-Buch',
 'Pyla-sur-Mer',
 'Gujan-Mestras',
 'Le Teich',
 'Biganos',
 'Audenge',
 'Lanton',
 'Andernos-les-Bains',
 'Arès',
 'Lège-Cap-Ferret',
 'Mios',
 'Marcheprime',
 'Salles',
 'Belin-Béliet',
 'Le Barp',
 'Lugos',
 'Saint-Magne']

## Construction du filtre `where`

Le filtre `refine` est pratique pour explorer des facettes, mais il n'est pas adapté pour exprimer proprement une condition de plage de dates.

Ici, on utilise donc `where` pour combiner :

- le département ;
- la liste des villes ;
- la contrainte temporelle : événements encore actifs ou à venir à partir de la date de référence du POC.

In [4]:
where_clause = build_openagenda_where_clause()

print(where_clause)

location_department = "Gironde"
AND location_region = "Nouvelle-Aquitaine"
AND location_countrycode IN ("FR", "fr")
AND location_city IN ("Arcachon", "La Teste-de-Buch", "Pyla-sur-Mer", "Gujan-Mestras", "Le Teich", "Biganos", "Audenge", "Lanton", "Andernos-les-Bains", "Arès", "Lège-Cap-Ferret", "Mios", "Marcheprime", "Salles", "Belin-Béliet", "Le Barp", "Lugos", "Saint-Magne")
AND lastdate_end >= date'2026-05-01'


## Premier appel API

On récupère un petit nombre d'événements pour vérifier que l'endpoint, les paramètres et le filtre fonctionnent.

In [5]:
params = {
    "limit": 20,
    "where": where_clause,
    "order_by": OPENAGENDA_ORDER_BY,
}

response = requests.get(OPENAGENDA_BASE_URL, params=params, timeout=30)
response.raise_for_status()

data = response.json()

data.keys()

dict_keys(['total_count', 'results'])

In [6]:
total_count = data.get("total_count", 0)
events = data.get("results", [])

print(f"Nombre total d'événements trouvés : {total_count}")
print(f"Nombre d'événements récupérés dans cette page : {len(events)}")

Nombre total d'événements trouvés : 138
Nombre d'événements récupérés dans cette page : 20


## Inspection rapide d'un événement

Cette cellule permet de visualiser la structure JSON brute renvoyée par l'API.

In [7]:
if events:
    sample_event = events[0]
    preview_fields = [
        "uid",
        "title_fr",
        "description_fr",
        "daterange_fr",
        "firstdate_begin",
        "lastdate_end",
        "location_name",
        "location_city",
        "canonicalurl",
    ]

    preview_event = {
        field: sample_event.get(field)
        for field in preview_fields
    }

    print(json.dumps(preview_event, ensure_ascii=False, indent=2))
else:
    print("Aucun événement trouvé avec ces filtres.")

{
  "uid": "13708573",
  "title_fr": "Initiation à l'astronomie à Lanton",
  "description_fr": "Initiation à l'astronomie et observation aux instruments.",
  "daterange_fr": "5 janvier 2026 - 3 janvier 2027",
  "firstdate_begin": "2026-01-05T19:30:00+00:00",
  "lastdate_end": "2027-01-03T21:30:00+00:00",
  "location_name": "Blagon, 33138 Lanton, France",
  "location_city": "Lanton",
  "canonicalurl": "https://openagenda.com/econature/events/initiation-a-lastronomie-a-lanton"
}


## Conversion en DataFrame

Le DataFrame facilite l'inspection des colonnes et des valeurs manquantes.

In [8]:
df = pd.DataFrame(events)

df.shape

(20, 56)

In [9]:
df.head()

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,13708573,initiation-a-lastronomie-a-lanton,https://openagenda.com/econature/events/initia...,Initiation à l'astronomie à Lanton,Initiation à l'astronomie et observation aux i...,<p>✨ Découvrez le ciel nocturne comme vous ne ...,Tarifs : Enfant -18 ans 10€ ; Adulte 18 ans et...,"[Nature, Sortie nature, astronomie]",https://cdn.openagenda.com/main/291a2ef7c21140...,NaN,...,99155778,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://www.eco-na...","[{""link"": ""https://www.eco-nature.org/experien..."
1,10772673,mai-a-velo-2026-1779336,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=209855],https://cdn.openagenda.com/main/9358618cf0ab46...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
2,49861420,mai-a-velo-2026-436875,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=202572],https://cdn.openagenda.com/main/bcca417c031c44...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
3,97562973,mai-a-velo-2026-23977,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=211745],https://cdn.openagenda.com/main/8ff0ec673d324a...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN
4,21183348,mai-a-velo-2026-8312624,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=213084],https://cdn.openagenda.com/main/7f2f220d9af143...,NaN,...,38376650,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://challenge-...",NaN


## Colonnes disponibles

On liste les champs retournés pour décider lesquels seront utiles dans le RAG.

In [10]:
list(df.columns)

['uid',
 'slug',
 'canonicalurl',
 'title_fr',
 'description_fr',
 'longdescription_fr',
 'conditions_fr',
 'keywords_fr',
 'image',
 'imagecredits',
 'thumbnail',
 'originalimage',
 'updatedat',
 'daterange_fr',
 'firstdate_begin',
 'firstdate_end',
 'lastdate_begin',
 'lastdate_end',
 'timings',
 'accessibility',
 'accessibility_label_fr',
 'location_uid',
 'location_coordinates',
 'location_name',
 'location_address',
 'location_district',
 'location_insee',
 'location_postalcode',
 'location_city',
 'location_department',
 'location_region',
 'location_countrycode',
 'location_image',
 'location_imagecredits',
 'location_phone',
 'location_website',
 'location_links',
 'location_tags',
 'location_description_fr',
 'location_access_fr',
 'attendancemode',
 'onlineaccesslink',
 'status',
 'age_min',
 'age_max',
 'originagenda_title',
 'originagenda_uid',
 'contributor_email',
 'contributor_contactnumber',
 'contributor_contactname',
 'contributor_contactposition',
 'contributor_organ

## Champs utiles pour le RAG

On garde uniquement les champs qui peuvent aider à recommander ou expliquer un événement : titre, description, dates, lieu, conditions, lien officiel, coordonnées.

In [11]:
available_useful_columns = [col for col in OPENAGENDA_USEFUL_FIELDS if col in df.columns]
missing_useful_columns = [col for col in OPENAGENDA_USEFUL_FIELDS if col not in df.columns]

print("Champs utiles disponibles :", len(available_useful_columns))
print("Champs utiles absents dans cet échantillon :", len(missing_useful_columns))

available_useful_columns

Champs utiles disponibles : 28
Champs utiles absents dans cet échantillon : 0


['uid',
 'slug',
 'canonicalurl',
 'title_fr',
 'description_fr',
 'longdescription_fr',
 'conditions_fr',
 'keywords_fr',
 'daterange_fr',
 'firstdate_begin',
 'firstdate_end',
 'lastdate_begin',
 'lastdate_end',
 'timings',
 'location_name',
 'location_address',
 'location_postalcode',
 'location_city',
 'location_department',
 'location_region',
 'location_countrycode',
 'location_coordinates',
 'accessibility_label_fr',
 'age_min',
 'age_max',
 'registration',
 'onlineaccesslink',
 'originagenda_title']

In [12]:
df_useful = df[available_useful_columns].copy()
df_useful.head()

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,daterange_fr,firstdate_begin,...,location_department,location_region,location_countrycode,location_coordinates,accessibility_label_fr,age_min,age_max,registration,onlineaccesslink,originagenda_title
0,13708573,initiation-a-lastronomie-a-lanton,https://openagenda.com/econature/events/initia...,Initiation à l'astronomie à Lanton,Initiation à l'astronomie et observation aux i...,<p>✨ Découvrez le ciel nocturne comme vous ne ...,Tarifs : Enfant -18 ans 10€ ; Adulte 18 ans et...,"[Nature, Sortie nature, astronomie]",5 janvier 2026 - 3 janvier 2027,2026-01-05T19:30:00+00:00,...,Gironde,Nouvelle-Aquitaine,FR,"{'lon': -0.934394, 'lat': 44.783281}",None,None,None,"[{""type"": ""link"", ""value"": ""https://www.eco-na...",None,EcoNature
1,10772673,mai-a-velo-2026-1779336,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=209855],1 - 31 mai,2026-04-30T22:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,fr,"{'lon': -0.938355, 'lat': 44.605771}",None,None,None,"[{""type"": ""link"", ""value"": ""https://challenge-...",None,Challenges Geovelo
2,49861420,mai-a-velo-2026-436875,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=202572],1 - 31 mai,2026-04-30T22:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,fr,"{'lon': -1.127944, 'lat': 44.616267}",None,None,None,"[{""type"": ""link"", ""value"": ""https://challenge-...",None,Challenges Geovelo
3,97562973,mai-a-velo-2026-23977,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=211745],1 - 31 mai,2026-04-30T22:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,fr,"{'lon': -1.101869, 'lat': 44.743788}",None,None,None,"[{""type"": ""link"", ""value"": ""https://challenge-...",None,Challenges Geovelo
4,21183348,mai-a-velo-2026-8312624,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à Vélo 2026,Faites parcourir à votre communauté le plus de...,NaN,NaN,[challenge-id=213084],1 - 31 mai,2026-04-30T22:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,fr,"{'lon': -1.087861, 'lat': 44.745758}",None,None,None,"[{""type"": ""link"", ""value"": ""https://challenge-...",None,Challenges Geovelo


## Vérification du filtre temporel

Le POC utilise une date de référence figée au 2026-05-01. Le filtre conserve les événements dont `lastdate_end` est supérieur ou égal à cette date.

In [13]:
if not df_useful.empty and "lastdate_end" in df_useful.columns:
    df_useful["lastdate_end"] = pd.to_datetime(df_useful["lastdate_end"], errors="coerce")
    print("Date minimale :", df_useful["lastdate_end"].min())
    print("Date maximale :", df_useful["lastdate_end"].max())
else:
    print("Aucun événement à vérifier.")

Date minimale : 2026-05-05 10:30:00+00:00
Date maximale : 2027-01-03 21:30:00+00:00


## Vérification du filtre géographique

On vérifie que les villes récupérées correspondent bien à la zone choisie.

In [14]:
if "location_city" in df_useful.columns:
    df_useful["location_city"].value_counts(dropna=False)
else:
    print("La colonne location_city n'est pas présente.")

## Pagination

L'API utilise `limit` pour le nombre de résultats et `offset` pour passer aux pages suivantes.

Cette fonction permet de récupérer plusieurs pages sans encore créer le client final.

In [15]:
def fetch_openagenda_page(limit=OPENAGENDA_PAGE_SIZE, offset=0):
    params = {
        "limit": limit,
        "offset": offset,
        "where": where_clause,
        "order_by": OPENAGENDA_ORDER_BY,
    }

    response = requests.get(OPENAGENDA_BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


page_1 = fetch_openagenda_page(limit=5, offset=0)
page_2 = fetch_openagenda_page(limit=5, offset=5)

print("Page 1 :", len(page_1.get("results", [])), "événements")
print("Page 2 :", len(page_2.get("results", [])), "événements")

Page 1 : 5 événements
Page 2 : 5 événements


## Sauvegarde d'un échantillon brut

In [16]:
sample_path = PATHS.data_raw / RAW_OPENAGENDA_SAMPLE_FILENAME
saved_path = save_openagenda_sample(data, sample_path)

print(f"Échantillon sauvegardé dans : data/raw/{saved_path.name}")

Échantillon sauvegardé dans : data/raw/sample_openagenda.json


## Synthèse des décisions

- Endpoint retenu : `/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records`.
- Aucune clé API nécessaire pour ce POC.
- Utilisation de `where` plutôt que `refine` pour les filtres métier.
- Filtre temporel retenu : `lastdate_end >= date'2026-05-01'`.
- Filtre géographique : département Gironde + villes ciblées.
- Champs utiles : titre, descriptions, conditions, dates, lieu, adresse, ville, coordonnées, URL.
- Problèmes observés : descriptions HTML, champs parfois manquants, certains champs JSON stockés sous forme de chaînes.
- La suite consistera à nettoyer les textes et construire un document textuel indexable pour le RAG.